In [8]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [9]:
########  VARIABLES Y FUNCIONES DE PERTENENCIA ########

### VARIABLES
x_calidad = np.arange(0, 11, 0.1)
x_espera = np.arange(0, 31, 0.1)
x_satisfaccion = np.arange(0, 11, 0.1)

### FUNCIONES
# (Triangulares)
calidad_baja = fuzz.trimf(x_calidad, [0, 0, 5])
calidad_media = fuzz.trimf(x_calidad, [3, 5, 7])
calidad_alta = fuzz.trimf(x_calidad, [5, 10, 10])
# (Trapezoidales)
espera_corta = fuzz.trapmf(x_espera, [0, 0, 5, 10])
espera_media = fuzz.trapmf(x_espera, [8, 12, 18, 22])
espera_larga = fuzz.trapmf(x_espera, [20, 25, 30, 30])
# (Gausianas)
satisfaccion_baja = fuzz.gaussmf(x_satisfaccion, 2, 1.5)
satisfaccion_media = fuzz.gaussmf(x_satisfaccion, 5, 1.5)
satisfaccion_alta = fuzz.gaussmf(x_satisfaccion, 8, 1.5)


########  ENTRADA DEL SISTEMA Y FUZZIFICACIÓN DE VARIABLES ########

nivel_calidad_baja = fuzz.interp_membership(x_calidad, calidad_baja, 3)
nivel_espera_larga = fuzz.interp_membership(x_espera, espera_larga, 25)


########  REGLAS DIFUSAS ########

calidad = ctrl.Antecedent(x_calidad, 'calidad')
espera = ctrl.Antecedent(x_espera, 'espera')
satisfaccion = ctrl.Consequent(x_satisfaccion, 'satisfaccion')

calidad['baja'] = calidad_baja
calidad['media'] = calidad_media
calidad['alta'] = calidad_alta

espera['corta'] = espera_corta
espera['media'] = espera_media
espera['larga'] = espera_larga

satisfaccion['baja'] = satisfaccion_baja
satisfaccion['media'] = satisfaccion_media
satisfaccion['alta'] = satisfaccion_alta

# satisfacción ALTA = Calidad ALTA Y espera CORTA
regla1 = ctrl.Rule(calidad['alta'] & espera['corta'], satisfaccion['alta'])
# satisfacción BAJA = Calidad BAJA O espera LARGA
regla2 = ctrl.Rule(calidad['baja'] | espera['larga'], satisfaccion['baja'])
# satisfacción MEDIA = Calidad MEDIA Y espera MEDIA
regla3 = ctrl.Rule(calidad['media'] & espera['media'], satisfaccion['media'])


########  APLICACIÓN DE REGLAS A CONJUNTO ########
#---------------------------------------------------------------------

sistema_ctrl = ctrl.ControlSystem([regla1, regla2, regla3])
sistema = ctrl.ControlSystemSimulation(sistema_ctrl)

print(f"--- RESULTADOS CON DATOS DE PRUEBA ---")

casos_prueba = [(3, 25),(9, 2),(6, 12)]

for cal, esp in casos_prueba:
    sistema.input['calidad'] = cal
    sistema.input['espera'] = esp
    sistema.compute()
    resultado = sistema.output['satisfaccion']
    print(f"Datos: Calidad: {cal}, Espera: {esp}")
    print(f"Satisfacción final: {resultado:.2f}\n")

--- RESULTADOS CON DATOS DE PRUEBA ---
Datos: Calidad: 3, Espera: 25
Satisfacción final: 2.27

Datos: Calidad: 9, Espera: 2
Satisfacción final: 7.90

Datos: Calidad: 6, Espera: 12
Satisfacción final: 5.00

